In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenizer = Tokenizer()

In [ ]:
with open('my_sentences.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

In [ ]:
tokenizer.fit_on_texts(lines)

In [ ]:
len(tokenizer.word_index)

**Create Dataset**
Input          Output
I              am
I am           writing
I am writing   to

In [ ]:
input_sequences = []
for sentence in lines:
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [ ]:
max_len = max([len(x) for x in input_sequences])

**Zero Padding**
Need as different line of different size

In [ ]:
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')

In [ ]:
padded_input_sequences

Input

In [ ]:
X = padded_input_sequences[:,:-1]

In [ ]:
X.shape

Output

In [ ]:
y = padded_input_sequences[:,-1]

In [ ]:
y.shape

OHE as we treat it like categorical

In [ ]:
y = to_categorical(y,num_classes=2031)

In [ ]:
y.shape

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
input_seq_length = X.shape[1]

model = Sequential()
model.add(Embedding(vocab_size, 100, input_length=input_seq_length))
model.add(LSTM(150))
model.add(LSTM(150))
model.add(Dense(vocab_size, activation='softmax'))

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
import time
import numpy as np
text = "I would like"

for i in range(10):
  token_text = tokenizer.texts_to_sequences([text])[0]
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

In [ ]:
!pip install keras-tuner
import keras_tuner as kt

Keras hypertuning

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
input_seq_length = X.shape[1]

def build_model(hp):
    model = Sequential()
    model.add(Embedding(vocab_size, 100, input_length=input_seq_length))

    # Tune the number of units in the first LSTM layer
    hp_units_lstm1 = hp.Int('units_lstm1', min_value=50, max_value=250, step=50)
    model.add(LSTM(units=hp_units_lstm1, return_sequences=True))

    # Tune the number of units in the second LSTM layer
    hp_units_lstm2 = hp.Int('units_lstm2', min_value=50, max_value=250, step=50)
    model.add(LSTM(units=hp_units_lstm2))

    model.add(Dense(vocab_size, activation='softmax'))

    # Tune the learning rate for the optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate)

    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

In [ ]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='next_word_prediction'
)

In [ ]:
tuner.search_space_summary()

In [ ]:
tuner.search(X, y, epochs=10, validation_split=0.2, batch_size=32)

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best LSTM1 units: {best_hps.get('units_lstm1')}")
print(f"Best LSTM2 units: {best_hps.get('units_lstm2')}")
print(f"Best learning rate: {best_hps.get('learning_rate')}")

best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
best_model.fit(X, y, epochs=90, validation_split=0.2, batch_size=32, initial_epoch=10, callbacks=[early_stopping])

Choose next word of highest probability

In [ ]:
import time
import numpy as np
text = "I would like"

for i in range(10):
  token_text = tokenizer.texts_to_sequences([text])[0]
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  pos = np.argmax(best_model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)